# 🔬 02 · What Is a Digital Image? Seeing Like a Microscope
### Optics, contrast, and sampling — how light becomes an array of numbers

**Part B · Seeing like a microscope** · Module 02 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/02_what_is_a_digital_image.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib, SciPy |
| **Prerequisites** | Modules 00–01 |
| **Estimated time** | 75–90 minutes |
| **02-740 topics covered** | Optics fundamentals; light vs. electron microscopy; contrast generation (bright-field/dark-field/phase/DIC/fluorescence); sampling, quantization, bit depth, calibration |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every microscope image in this notebook is a SIMULATED array built from an optical model; no real microscope file is ever read.

## 🎯 Learning objectives

1. State the laws of reflection and refraction, and explain why they let a lens magnify.
2. Explain why light microscopy and electron microscopy occupy different resolution regimes, and
   what trade-off that forces.
3. Describe at least four ways microscopes generate contrast (bright-field, dark-field, phase
   contrast, fluorescence) and why each exists.
4. Explain sampling, quantization, bit depth, and pixel-size calibration, and connect each to a
   concrete consequence for image analysis.
5. Simulate a simple point-spread function (PSF) and use it to blur a synthetic image, the way a
   real microscope's optics blur a real sample.

## 🗺️ Contents

1. Basic optics: reflection, refraction, and why lenses magnify
2. Light microscopy vs. electron microscopy
3. Generating contrast: bright-field, dark-field, phase contrast, fluorescence
4. Sampling, quantization, bit depth, and calibration
5. The point-spread function: why every microscope image is already blurred

## 🧫 The biological question

A living cell is, to visible light, almost perfectly transparent. Unstained, unlabeled cells barely
absorb any light at all, which is why a plain bright-field microscope shows you almost nothing
interesting about a living cell's interior — the light passes straight through with only the
tiniest change. And yet biologists routinely produce strikingly detailed, information-rich images
of living cells: the mitotic spindle pulling chromosomes apart, actin networks reshaping a cell's
leading edge, individual vesicles moving along an axon. How?

The answer is that *every* microscopy technique is a deliberate trick for converting some otherwise
invisible physical property of the sample — a tiny phase shift in light, the presence of a specific
tagged protein, the density of electrons — into something a detector (or an eye) can register as a
difference in brightness or color. Understanding which trick is being used in a given image is not
a side note: it directly determines what the pixel values in that image actually *mean*, and
therefore what analysis is valid to do on them.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 2
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · Basic optics: reflection, refraction, and why lenses magnify

Two laws from introductory physics underlie every lens in every microscope:

- **Law of reflection**: the angle of incidence equals the angle of reflection, $\theta_i = \theta_r$.
- **Law of refraction (Snell's law)**: $n_i \sin\theta_i = n_t \sin\theta_t$, where $n$ is the
  refractive index of a material (how much slower light travels in it than in a vacuum: $n = c / v$).

*In words:* when light crosses a boundary between two materials of different refractive index (say,
air into glass), it bends. A lens is simply a precisely curved piece of glass that uses this bending
to bring parallel rays of light to a single focus point — magnifying whatever sits at that focus. A
light microscope's objective lens and eyepiece bend the light twice in sequence, producing the
overall magnification.

In [ ]:
# ── 1.1 a minimal ray-bending simulation illustrating refraction ──
n_air, n_glass = 1.0003, 1.52
theta_i_deg = np.linspace(1, 80, 80)
theta_i = np.deg2rad(theta_i_deg)
theta_t = np.arcsin(np.clip((n_air / n_glass) * np.sin(theta_i), -1, 1))  # Snell's law, solved for theta_t

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(theta_i_deg, np.rad2deg(theta_t), color='teal')
ax.plot(theta_i_deg, theta_i_deg, '--', color='gray', alpha=0.6, label='no bending (theta_t = theta_i)')
ax.set_xlabel('angle of incidence in air (degrees)'); ax.set_ylabel('angle of refraction in glass (degrees)')
ax.set_title("Snell's law: light bends toward the normal entering a denser medium")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** As light goes from air ($n \approx 1.0$) into glass ($n \approx 1.5$,
a *denser* optical medium), the refracted ray bends *toward* the boundary's normal — the refraction
angle is always smaller than the incidence angle. This is precisely the bending a microscope
objective lens exploits, over and over through its stack of lens elements, to bring light from a
tiny specimen to a sharp, magnified focus.

✏️ **Your turn.** Change `n_glass` to `1.33` (the refractive index of water, close to that of a
living cell) and re-run. Does the ray bend more or less than with glass? Immersion oil, used between
some high-magnification objectives and the sample, has $n \approx 1.51$ — almost identical to glass
— specifically to minimize unwanted bending at that interface. Can you now explain, in one sentence,
why that minimizes image-degrading refraction?

---
## 2 · Light microscopy vs. electron microscopy

In [ ]:
# ── 2.1 resolution vs. wavelength: why electrons can "see" much smaller things ──
# Rough diffraction-limited resolution ~ wavelength / 2 (a simplification of the Abbe/Rayleigh limit)
wavelengths_nm = {'visible light (red)': 650, 'visible light (violet)': 400, 'electrons (80 kV TEM, De Broglie)': 0.0042}
for name, wl in wavelengths_nm.items():
    print(f'{name:38s}: wavelength ~ {wl:>8.4f} nm  ->  resolution limit roughly {wl/2:>9.4f} nm')

🔍 **What just happened?** The resolution limit of any imaging technique that uses waves is set,
fundamentally, by the wavelength of the wave used — shorter wavelength means finer resolvable
detail. Visible light's wavelength (roughly 400–750 nm) limits conventional light microscopy to
structures larger than about 200 nm; electrons, because of their much smaller effective (de Broglie)
wavelength, allow electron microscopy to resolve structures below 1 nm — a difference of several
orders of magnitude. This single physical fact is why light and electron microscopy occupy
genuinely different, complementary roles in biology rather than one simply being a "better version"
of the other.

### 📘 Lecture notes — light microscopy vs. electron microscopy

**Definition.** Light microscopy uses visible light (wavelength roughly 400–750 nm) and resolves
structures above roughly 200 nm; it does little damage to a sample and can image living cells at
room temperature. Electron microscopy uses a focused beam of electrons (far shorter effective
wavelength) and resolves structures down to about 0.1 nm, but requires the sample to be chemically
fixed and imaged in a vacuum, since electrons damage living tissue and cannot travel far through air.

**Worked example.** Imaging actin networks at the leading edge of a migrating cell under light
microscopy shows them as diffraction-limited diffuse structures that must be inferred rather than
directly resolved; the same network imaged by electron microscopy shows individual filaments as
sharp line structures, but only in a fixed, no-longer-living sample.

**One memorable point.** *Light microscopy trades resolution for the ability to watch a living
process unfold; electron microscopy trades the ability to watch a living process for far higher
resolution. Neither technique dominates the other — they answer different questions.*

---
## 3 · Generating contrast: bright-field, dark-field, phase contrast, fluorescence

Every microscope has two fundamental jobs: provide adequate **resolution** (distinguish two
close-together points as separate) and provide adequate **contrast** (make an object distinguishable
from its background at all). Resolution without contrast is useless — you could in principle
resolve a transparent cell's internal structures perfectly and still see nothing, because nothing
in the image differs in brightness.

- **Bright-field**: light transmitted straight through the sample; dense regions absorb more light
  and appear dark against a bright background. Simple, but gives almost no contrast for transparent,
  unstained living cells.
- **Dark-field**: the unscattered, direct beam is blocked from reaching the detector, so only light
  *scattered* by the sample reaches the image — the background goes dark, and small
  scattering objects (thin filaments, small particles) appear as bright points against it.
- **Phase contrast**: converts invisible phase shifts (caused by a sample's refractive index
  differing slightly from its surroundings) into visible brightness differences — ideal for
  transparent, unstained living cells, at the cost of some artifacts (halos around edges).
- **Fluorescence**: a completely different strategy — rather than relying on the sample's own
  optical properties, a specific molecule (a fluorophore, such as GFP) is excited by light of one
  wavelength and re-emits light at a longer wavelength, which is then selectively collected. This
  gives extremely high contrast and, crucially, **molecular specificity**: you see exactly where
  your labeled protein is, and (mostly) nothing else.

In [ ]:
# ── 3.1 simulate the SAME underlying sample under different contrast mechanisms ──
size = 140
rng_local = np.random.default_rng(SEED)
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
cy, cx, r = size / 2, size / 2, size * 0.22
refractive_bump = np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * (r * 0.6) ** 2)))  # the "cell", as a smooth bump

# Bright-field: dense (high refractive_bump) regions absorb light -> appear DARKER
background_level = 0.85
brightfield = background_level - 0.5 * refractive_bump
brightfield = add_gaussian_noise(brightfield, 0.02, rng_local)

# Dark-field: only the EDGES (where the bump changes fastest = scatters light) show up, on a dark background
grad_y, grad_x = np.gradient(refractive_bump)
darkfield = np.clip(np.hypot(grad_y, grad_x) * 8, 0, 1)
darkfield = add_gaussian_noise(darkfield, 0.015, rng_local)

# Phase contrast: contrast proportional to the gradient of optical path length, with some signed shading
phase = 0.5 + 0.9 * (grad_x + grad_y) * 5
phase = np.clip(phase, 0, 1)
phase = add_gaussian_noise(phase, 0.02, rng_local)

# Fluorescence: ONLY the labeled structure glows; everything else is black (high specificity, high contrast)
fluorescence = np.clip(refractive_bump * 1.2, 0, 1)
fluorescence = add_gaussian_noise(fluorescence, 0.01, rng_local)
fluorescence = np.clip(fluorescence, 0, None)

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, (title, arr) in zip(axes, [('Bright-field', brightfield), ('Dark-field', darkfield),
                                    ('Phase contrast', phase), ('Fluorescence', fluorescence)]):
    show_image(ax, arr, title)
fig.suptitle('The SAME simulated cell, four different contrast mechanisms', y=1.04)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** All four panels above were generated from the identical
`refractive_bump` — the exact same underlying "cell". Any differences you see between the panels
are purely a consequence of which optical contrast mechanism we modeled, not a different sample —
exactly the point this section makes: the imaging technique itself shapes what you see at least as
much as the sample does.

✏️ **Your turn.** The fluorescence panel has by far the highest contrast relative to its (near
zero) background. In your own words, explain why this makes fluorescence microscopy especially
well suited to detecting a small number of specifically labeled molecules against a complex,
otherwise-unlabeled cellular background — connect this to the axonal-transport-vesicle motivating
example from Module 00.

---
## 4 · Sampling, quantization, bit depth, and calibration

Two more conversions happen between "a continuous optical image" and "a NumPy array":

- **Sampling**: the continuous image is measured only at a discrete grid of sensor pixel locations.
  The **Nyquist–Shannon sampling theorem** says that to capture all the information in a signal with
  no frequency content above $B$ Hz, you must sample at a rate of at least $2B$ Hz — undersampling
  below this rate causes **aliasing**, where fine detail is misrepresented as a completely different,
  coarser pattern. In microscopy this means your pixel size must be small enough, relative to the
  optical resolution, to avoid throwing away real structural information.
- **Quantization**: each sampled intensity is rounded to one of a finite number of discrete levels —
  the **bit depth**. An 8-bit image has $2^8 = 256$ possible gray levels (0–255); a 12- or 16-bit
  image (common in scientific cameras) has 4096 or 65536 levels, capturing much finer brightness
  differences, which matters enormously when quantifying faint fluorescence signals.
- **Calibration**: a pixel is not a biological unit — you must know the **pixel size** (how many
  micrometers or nanometers one pixel edge corresponds to, which itself depends on magnification),
  typically from the relation $p = \frac{x}{y}$ scaled by the microscope's magnification, in order
  to convert a measurement in pixels into a measurement in real physical units.

In [ ]:
# ── 4.1 simulate the effect of bit depth (quantization) on a smooth gradient ──
smooth = np.linspace(0, 1, 256)[None, :].repeat(40, axis=0)  # a smooth horizontal gradient

def quantize(image, n_bits):
    levels = 2 ** n_bits
    return np.floor(image * (levels - 1)) / (levels - 1)

fig, axes = plt.subplots(4, 1, figsize=(7, 5.5))
for ax, bits in zip(axes, [8, 4, 2, 1]):
    ax.imshow(quantize(smooth, bits), cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'{bits}-bit ({2**bits} gray levels)', fontsize=9); ax.set_yticks([]); ax.set_xticks([])
plt.tight_layout(); plt.show()

🔍 **What just happened?** At 8 bits (256 levels), the gradient looks smooth to the eye. By 2
bits (4 levels) you can clearly see "banding" — hard steps where there should be a continuous
change. Real scientific cameras typically use 12 or 16 bits specifically so that subtle intensity
differences (for instance, a weak fluorescence signal just above background) are not lost to
quantization before any analysis even begins.

In [ ]:
# ── 4.2 aliasing: what happens when you undersample a fine periodic structure ──
true_period_px = 3.0    # the real structure repeats every 3 "continuous" pixels
x_fine = np.linspace(0, 60, 2000)
true_signal = np.sin(2 * np.pi * x_fine / true_period_px)

for sample_spacing in [1.0, 2.9]:   # 1.0 = well above Nyquist; 2.9 = badly undersampled
    x_samples = np.arange(0, 60, sample_spacing)
    y_samples = np.sin(2 * np.pi * x_samples / true_period_px)
    fig, ax = plt.subplots(figsize=(7, 1.8))
    ax.plot(x_fine, true_signal, color='lightgray', lw=1, label='true continuous structure')
    ax.plot(x_samples, y_samples, 'o-', color='crimson', markersize=3, label=f'sampled every {sample_spacing} px')
    ax.set_title(f'Sample spacing = {sample_spacing} px (period = {true_period_px} px)', fontsize=9)
    ax.legend(fontsize=7, loc='upper right'); ax.set_xlim(0, 30)
    plt.tight_layout(); plt.show()

🔍 **What just happened?** When sampling is fine relative to the structure's period (1.0 px
spacing for a 3.0 px period), the samples faithfully trace the true oscillation. When the sample
spacing (2.9 px) is close to the structure's own period, the red sampled points trace out a much
*slower*, completely different-looking oscillation than the true signal — this is **aliasing**, and
it is exactly why Nyquist sampling matters: with too coarse a pixel size, a real fine structure can
be misrepresented as something else entirely, not just blurred away.

---
## 5 · The point-spread function: why every microscope image is already blurred

No real lens is perfect. Even a perfectly in-focus point source of light is imaged by a real
microscope not as a single point, but as a small blurred spot — the microscope's **point-spread
function (PSF)**. Every real image is, to a good first approximation, the "true" scene convolved
with the microscope's PSF, plus noise. This is why, starting right now, every synthetic bioimage in
this course is built by taking a sharp, "ground truth" structure and deliberately blurring it with a
PSF and adding noise — simulating, honestly, what a real microscope would hand you.

In [ ]:
# ── 5.1 a Gaussian approximation of a microscope PSF, and what it does to a sharp image ──
sim = simulate_nuclei_image(n_cells=10, size=120, seed=SEED)
sharp_ground_truth = (sim['instance_mask'] > 0).astype(float)   # a perfectly sharp binary "ground truth"

for sigma in [0.5, 1.5, 3.0]:
    psf = gaussian_psf(sigma)
    blurred = blur_image(sharp_ground_truth, psf)
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    show_image(axes[0], sharp_ground_truth, 'Sharp ground truth')
    show_image(axes[1], blurred, f'Blurred by PSF, sigma={sigma}')
    plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** `sharp_ground_truth` is a perfectly sharp binary mask — this is
information we, as simulation authors, know exactly and could never obtain this cleanly from a real
microscope image. The blurred versions are what a (simulated) microscope would actually hand an
experimenter. This gap between "ground truth" and "what the microscope gives you" is the entire
reason restoration and deconvolution (Module 25) and denoising (Module 24) exist as fields.

✏️ **Your turn.** Using `simulate_nuclei_image(..., seed=42)` with a different seed, generate a
new synthetic nuclei image and blur it with `sigma=2.0`. Then add Gaussian noise with
`add_gaussian_noise(blurred, 0.05, rng)`. Display the sharp, blurred, and blurred-plus-noisy versions
side by side using `show_image_grid`.

### 📘 Lecture notes — the point-spread function

**Definition.** The point-spread function (PSF) describes how a microscope images an idealized
single point of light; because microscope optics are (to a very good approximation) linear and
shift-invariant, the image of an entire extended sample is modeled as the convolution of the true
scene with the PSF, plus additive noise: $\text{image} \approx \text{PSF} * \text{scene} +
\text{noise}$.

**Worked example.** A larger PSF (e.g., from a lower numerical-aperture objective, or from being out
of focus) blurs two nearby point-like structures together until they can no longer be distinguished
as separate — this is the practical, image-formation-level meaning of "resolution limit".

**One memorable point.** *Every image-restoration and super-resolution method in this course
(Modules 24, 25, 28) is, at its core, an attempt to partially undo this one equation.*

## 📌 Key takeaways

- Resolution and contrast are two separate requirements; a microscope must provide both to make a biological structure usable.
- Light and electron microscopy occupy genuinely different resolution/live-imaging regimes because of their fundamentally different wavelengths.
- Different contrast mechanisms (bright-field, dark-field, phase contrast, fluorescence) convert different physical properties of the sample into visible brightness — know which one produced your image before interpreting its pixel values.
- Sampling (Nyquist), quantization (bit depth), and calibration (pixel size) all shape what information actually survives into the array you analyze.
- Every real microscope image is already blurred by the instrument's point-spread function — 'sharp' is an idealization that restoration algorithms chase but can never fully achieve.

## ✅ Checkpoint

**1.** Why does fluorescence microscopy achieve such high contrast compared to bright-field imaging
of an unstained living cell?

<details><summary>Answer</summary>
Because it does not rely on the sample's own weak light-absorption or refraction properties at all;
instead, a specific fluorophore is excited at one wavelength and emits at another, so (ideally) only
the labeled structure produces signal, against an otherwise dark background.
</details>

**2.** What goes wrong if an image is sampled at less than the Nyquist rate for the finest structure
present?

<details><summary>Answer</summary>
Aliasing: the fine structure is not just blurred away but can be misrepresented as a different,
coarser pattern that was never actually present in the sample.
</details>

**3.** Why do we model a microscope image as the true scene convolved with a PSF, rather than as an
exact copy of the scene?

<details><summary>Answer</summary>
Because no real lens is perfect — even a single point of light is spread out into a small blurred
spot by diffraction and optical imperfections, and this same spreading is applied (approximately,
by linearity) to every point in the sample.
</details>

## 🧠 Questions & discussion

- If you were designing an experiment to image a live, fast-moving process versus solving the
  highest-resolution structure of a fixed macromolecular complex, which microscopy technique would
  you reach for in each case, and why?
- Phase contrast can produce "halo" artifacts around object boundaries. Why might this matter for an
  automated segmentation algorithm, even if it looks only mildly distracting to a human eye?
- Why might a 16-bit camera be preferred over an 8-bit camera even though most computer monitors can
  only display 256 distinguishable gray levels?

## 🚀 Bonus — going further

- Simulate an "out of focus" image by using a larger PSF sigma only in a sub-region of the image
  (mimicking a sample that is not perfectly flat).
- Read about Differential Interference Contrast (DIC) microscopy and explain, in your own words, how
  it differs from phase contrast in what physical quantity it visualizes.
- Look up a maximum-intensity projection of a z-stack (a 3-D stack of 2-D images taken at different
  focal depths) and explain what question it is trying to answer that a single 2-D image cannot.

## 📝 Lab note

```text
Module: 02 — What Is a Digital Image?
Date:
Contrast mechanism I found most surprising:
A measurement I would NOT trust from a phase-contrast image, and why:
```

## 📚 References

1. Hecht, E. (2016). *Optics* (5th ed.). Pearson.
2. Inoué, S., Spring, K.R. (1997). *Video Microscopy: The Fundamentals* (2nd ed.). Springer.
3. Zernike, F. (1955). How I discovered phase contrast. *Science*, 121(3141), 345–349.
4. Shimomura, O. (2009). Discovery of green fluorescent protein (GFP) (Nobel Lecture). *Angewandte
   Chemie International Edition*, 48(31), 5590–5602.
5. Pawley, J.B. (ed.) (2006). *Handbook of Biological Confocal Microscopy* (3rd ed.). Springer.
6. Shannon, C.E. (1949). Communication in the presence of noise. *Proceedings of the IRE*, 37(1),
   10–21.
7. Goodman, J.W. (2005). *Introduction to Fourier Optics* (3rd ed.). Roberts and Company.

## ➡️ Next

Module 11 starts turning raw, noisy, blurred images into cleaner ones: convolution, kernels, and
the classical filtering toolbox that underlies nearly every later algorithm in this course.